# ABP trajectories: MUCA evolution and physical-time movies

One **MUCA movie frame is a complete physical trajectory** at a saved MCMC state.
The lower panel tracks its endpoint as the Markov chain explores path space.
The separate physical-time movie follows one trajectory from $t=0$ to its last
saved integration point. MCMC moves and physical time are different clocks.

Set `PHASE="muca"` for adaptive weight learning or `"production"` for the fixed
learned bias. Adaptive snapshots are diagnostic examples, with no physical
reweighting assigned. Production snapshots and overlays are also capped examples;
use the reweighted histograms to estimate probabilities.

New runs export `muca_movie_trajectories_long.csv` and/or
`movie_trajectories_long.csv`. Set `MOVIE_CSV` to an exact file, or use discovery
below. Old production CSVs are supported. No frame cap or roundtrip cropping is
applied by default. At 20 fps, 5,000 saved trajectories give about 250 seconds.
To retain all physical integration points, run with `ABP_MOVIE_PATH_TIME_THIN=1`.

In [ ]:
from pathlib import Path
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter

PROJECT_DIR = Path.cwd()
# PROJECT_DIR = Path("/home/nst/amarin/Desktop/ABP_Rare_Events/ABPNoiseSweep")
MOVIE_RUN_DIR = None  # Optional run folder; None searches PROJECT_DIR / "data".
MOVIE_CSV = None      # Optional exact CSV; takes priority over folder discovery.
PHASE = "muca"       # "muca" or "production"; no silent phase substitution.
CASE_CONTAINS = None # e.g. "D0p025"; None chooses the most recently written case.
CHAIN_ID = 1

SAVE_MOVIES = True
SAVE_FORMAT = "mp4"  # mp4 needs ffmpeg on PATH; gif uses Pillow.
FPS = 20
DPI = 120
MARKOV_FRAME_THIN = 1
MAX_MARKOV_FRAMES = None  # Optional evenly spaced cap, retaining first/last states.
PHYSICAL_MOVIE_ID = None  # None chooses the path with largest endpoint x.
PHYSICAL_FRAME_THIN = 1
N_OVERLAY_PATHS = 24
X_LIM = (-1.5, 1.5)  # Set None to show the complete saved extent.
Y_LIM = (-2.5, 2.5)
SHOW_LANDSCAPE = True  # Existing potential, only used as a display background.
ARROW_SCALE = 0.18

In [ ]:
def find_movie_csv():
    if PHASE not in {"muca", "production"}:
        raise ValueError("PHASE must be 'muca' or 'production'")
    if MOVIE_CSV is not None:
        result = Path(MOVIE_CSV).expanduser()
        if not result.is_absolute():
            result = PROJECT_DIR / result
        known_phase = {"muca_movie_trajectories_long.csv": "muca",
                       "movie_trajectories_long.csv": "production"}.get(result.name)
        if known_phase is not None and known_phase != PHASE:
            raise ValueError(f"Selected CSV is {known_phase}; set PHASE accordingly.")
        return result
    root = Path(MOVIE_RUN_DIR) if MOVIE_RUN_DIR is not None else PROJECT_DIR / "data"
    name = "muca_movie_trajectories_long.csv" if PHASE == "muca" else "movie_trajectories_long.csv"
    files = [p for p in root.rglob(name)
             if CASE_CONTAINS is None or CASE_CONTAINS in str(p.parent)]
    if not files:
        switch = "ABP_SAVE_MUCA_MOVIE_PATHS" if PHASE == "muca" else "ABP_SAVE_MOVIE_PATHS"
        raise FileNotFoundError(f"No {name} below {root}. Enable {switch}=true for a new run, "
                                "or select an existing production CSV with PHASE='production'.")
    return max(files, key=lambda p: p.stat().st_mtime)


def read_movie_csv(path):
    required = {"movie_id", "chain_id", "mcmc_step", "path_index", "physical_time",
                "x", "y", "theta", "endpoint_x", "endpoint_y"}
    optional = {"iteration", "iteration_step", "sampling_step"}
    # Read only plotting columns and the chosen chain. Diagnostic NaN weights in
    # adaptive MUCA files never cause otherwise valid trajectories to be dropped.
    chunks = []
    for chunk in pd.read_csv(path, usecols=lambda c: c in required | optional | {"phase"}, chunksize=200_000):
        if "phase" in chunk and not chunk.phase.eq(PHASE).all():
            raise ValueError(f"CSV phase does not match PHASE={PHASE!r}")
        missing = required - set(chunk.columns)
        if missing:
            raise ValueError(f"Missing CSV columns: {sorted(missing)}")
        chunk = chunk[chunk["chain_id"] == CHAIN_ID]
        if not chunk.empty:
            chunks.append(chunk)
    if not chunks:
        raise ValueError(f"No snapshots for CHAIN_ID={CHAIN_ID} in {path}")
    data = pd.concat(chunks, ignore_index=True)
    for name in required | optional:
        if name not in data:
            data[name] = 0 if name == "iteration" else data["mcmc_step"]
        data[name] = pd.to_numeric(data[name], errors="raise")
    if not np.isfinite(data[list(required)].to_numpy(dtype=float)).all():
        raise ValueError("Movie plotting columns contain nonfinite values")
    return data.sort_values(["mcmc_step", "movie_id", "path_index"]).reset_index(drop=True)


def keep_endpoints(n, stride=1, maximum=None):
    if stride < 1 or (maximum is not None and maximum < 1):
        raise ValueError("Frame thinning and frame budgets must be positive")
    # A one-frame budget can only show the final saved state.
    if maximum == 1:
        return np.array([n - 1], dtype=int) if n else np.array([], dtype=int)
    indices = np.arange(0, n, stride, dtype=int)
    if n and (not len(indices) or indices[-1] != n - 1):
        indices = np.append(indices, n - 1)
    if maximum is not None and len(indices) > maximum:
        indices = indices[np.unique(np.linspace(0, len(indices)-1, maximum).round().astype(int))]
    return indices


def movie_writer():
    if FPS <= 0:
        raise ValueError("FPS must be positive")
    if SAVE_FORMAT == "mp4":
        if shutil.which("ffmpeg") is None:
            raise RuntimeError("Install ffmpeg on PATH for MP4, or set SAVE_FORMAT='gif'.")
        return FFMpegWriter(fps=FPS, bitrate=2500)
    if SAVE_FORMAT == "gif":
        return PillowWriter(fps=FPS)
    raise ValueError("SAVE_FORMAT must be 'mp4' or 'gif'")


def landscape(ax):
    def extent(values):
        lo, hi = float(values.min()), float(values.max())
        pad = 0.05 * max(hi - lo, 1.0)
        return lo - pad, hi + pad
    x_limits = X_LIM if X_LIM is not None else extent(df.x)
    y_limits = Y_LIM if Y_LIM is not None else extent(df.y)
    if SHOW_LANDSCAPE:
        xx = np.linspace(*x_limits, 180)
        yy = np.linspace(*y_limits, 180)
        x, y = np.meshgrid(xx, yy)
        f0, fb = 0.5 * 2.5 * y**2, 1 + 0.5 * 0.3 * y**2
        fm = f0 * fb / (f0 + fb)
        potential = (x - 1)**2 * (x + 1)**2 * (fb - fm) + fm
        ax.contour(x, y, potential, levels=[0.05, 0.15, 0.3, 0.6, 1, 2],
                   colors="0.75", linewidths=0.7)
    ax.axvline(0, color="0.5", ls="--", lw=0.8)
    ax.set_xlim(*x_limits)
    ax.set_ylim(*y_limits)
    ax.set(xlabel="x(t)", ylabel="y(t)")

In [ ]:
csv_path = find_movie_csv()
df = read_movie_csv(csv_path)
summary = (df.groupby("movie_id", as_index=False)
           .agg(mcmc_step=("mcmc_step", "first"), endpoint_x=("endpoint_x", "first"),
                iteration=("iteration", "first"), n_points=("path_index", "count"))
           .sort_values("mcmc_step").reset_index(drop=True))
summary_movie = summary.iloc[keep_endpoints(len(summary), MARKOV_FRAME_THIN, MAX_MARKOV_FRAMES)]
# Group once instead of rescanning the long dataframe for every animation frame.
paths = {int(key): group.sort_values("path_index")
         for key, group in df.groupby("movie_id", sort=False)}
movie_out_dir = csv_path.parent / "movies"
if SAVE_MOVIES:
    movie_out_dir.mkdir(parents=True, exist_ok=True)
print(f"Input: {csv_path}")
print(f"Phase: {PHASE}; chain: {CHAIN_ID}; snapshots: {len(summary)}; "
      f"movie frames: {len(summary_movie)}; duration: {len(summary_movie)/FPS:.1f} s")
print(f"Attempted MCMC steps: {summary.mcmc_step.iloc[0]:g} to {summary.mcmc_step.iloc[-1]:g}")
print(f"Endpoint range: {summary.endpoint_x.min():.3g} to {summary.endpoint_x.max():.3g}")

In [ ]:
def make_markov_movie():
    frames = summary_movie.reset_index(drop=True)
    steps = frames.mcmc_step.to_numpy()
    endpoints = frames.endpoint_x.to_numpy()
    fig, (ax, trace) = plt.subplots(2, 1, figsize=(7, 7),
                                  gridspec_kw={"height_ratios": [3, 1]})
    landscape(ax)
    ax.set_title(f"{PHASE.upper()}: one complete trajectory per saved Markov state")
    line, = ax.plot([], [], lw=1.5, color="C0")
    point, = ax.plot([], [], "s", color="C1", ms=5)
    text = ax.text(0.02, 0.98, "", transform=ax.transAxes, va="top", fontsize=9)
    trace.plot(summary.mcmc_step, summary.endpoint_x, lw=0.8, color="0.5")
    marker, = trace.plot([], [], "o", ms=5)
    cursor = trace.axvline(steps[0], color="C0", lw=1)
    trace.set(xlabel="Attempted moves in selected chain", ylabel="Endpoint x(T)")
    fig.tight_layout()

    def update(i):
        row = frames.iloc[i]
        path = paths[int(row.movie_id)]
        line.set_data(path.x, path.y)
        point.set_data([path.x.iloc[-1]], [path.y.iloc[-1]])
        marker.set_data([row.mcmc_step], [row.endpoint_x])
        cursor.set_xdata([row.mcmc_step, row.mcmc_step])
        iteration = f"; MUCA iteration {int(row.iteration)}" if row.iteration > 0 else ""
        text.set_text(f"Frame {i+1}/{len(frames)}; chain {CHAIN_ID}{iteration}\n"
                      f"Attempted move {int(row.mcmc_step)}; x(T)={row.endpoint_x:.3f}")
        return line, point, text, marker, cursor

    anim = FuncAnimation(fig, update, frames=len(frames), interval=1000/FPS,
                         blit=False, cache_frame_data=False)
    output = movie_out_dir / f"{PHASE}_chain_{CHAIN_ID}_trajectory_evolution.{SAVE_FORMAT}"
    anim.save(output, writer=movie_writer(), dpi=DPI)
    plt.close(fig)
    return output


if SAVE_MOVIES:
    print("Saved:", make_markov_movie())

In [ ]:
def make_physical_movie():
    chosen = int(PHYSICAL_MOVIE_ID) if PHYSICAL_MOVIE_ID is not None else int(
        summary.iloc[summary.endpoint_x.to_numpy().argmax()].movie_id)
    if chosen not in paths:
        raise ValueError(f"PHYSICAL_MOVIE_ID={chosen} not found in selected chain")
    path = paths[chosen]
    indices = keep_endpoints(len(path), PHYSICAL_FRAME_THIN)
    x, y = path.x.to_numpy(), path.y.to_numpy()
    theta, times = path.theta.to_numpy(), path.physical_time.to_numpy()
    fig, ax = plt.subplots(figsize=(7, 5.5))
    landscape(ax)
    ax.set_title(f"One {PHASE} snapshot evolving in physical time")
    line, = ax.plot([], [], lw=1.5)
    point, = ax.plot([], [], "o", ms=4)
    arrow = ax.quiver([x[0]], [y[0]], [ARROW_SCALE*np.cos(theta[0])],
                      [ARROW_SCALE*np.sin(theta[0])], angles="xy", scale_units="xy", scale=1)
    text = ax.text(0.02, 0.98, "", transform=ax.transAxes, va="top")

    def update(k):
        i = indices[k]
        line.set_data(x[:i+1], y[:i+1])
        point.set_data([x[i]], [y[i]])
        arrow.set_offsets([[x[i], y[i]]])
        arrow.set_UVC([ARROW_SCALE*np.cos(theta[i])], [ARROW_SCALE*np.sin(theta[i])])
        text.set_text(f"Snapshot {chosen}; physical t={times[i]:.3f}\n"
                      f"Retained trajectory ends at t={times[-1]:.3f}")
        return line, point, arrow, text

    anim = FuncAnimation(fig, update, frames=len(indices), interval=1000/FPS,
                         blit=False, cache_frame_data=False)
    output = movie_out_dir / f"{PHASE}_snapshot_{chosen}_physical_trajectory.{SAVE_FORMAT}"
    anim.save(output, writer=movie_writer(), dpi=DPI)
    plt.close(fig)
    return output


if SAVE_MOVIES:
    print("Saved:", make_physical_movie())

## Inspect many complete saved trajectories

The overlay shows evenly spaced **saved states** across the selected Markov chain.
It preserves each trajectory from its initial point to its final saved point.
Plot limits only crop the display; set `X_LIM=Y_LIM=None` to see every saved point.
The production analysis notebook also shows separate $x(t)$, $y(t)$, and
$\theta(t)$ sequences from `saved_trajectories`, with no occupation-point filter.

In [ ]:
indices = keep_endpoints(len(summary), 1, N_OVERLAY_PATHS)
fig, (ax, ax_theta) = plt.subplots(1, 2, figsize=(12, 5))
landscape(ax)
for k in indices:
    movie_id = int(summary.iloc[k].movie_id)
    path = paths[movie_id]
    line, = ax.plot(path.x, path.y, lw=0.8, alpha=0.65)
    ax_theta.plot(path.physical_time, np.mod(path.theta, 2*np.pi), ".", ms=1.2,
                  color=line.get_color(), alpha=0.5)
ax.set_title(f"{len(indices)} complete saved {PHASE} trajectories")
ax_theta.set(xlabel="Physical time", ylabel=r"$\theta(t)$ modulo $2\pi$ [rad]")
fig.tight_layout()
plt.show()